- 这个脚本的目的是比较原来的采样方法和修正过后的采样方法在采样范围上的差别
- 原来的采样方法：针对整个底物6A范围内的表面网格进行采样
- 修改后的采样方法：可以将底物看作UDP和糖两部分，在两部分上分别采用不同的采样距离
- ！！！需要注意的是，采样距离的修改不需要重复Step1的计算，只需要在Step2的地方修改采样距离就可以实现平行。

# 结构对齐
- 调用之前开发的代码

In [ ]:
# # ==========脚本内容：Protein_align.py==========
# import subprocess
# import os
# from tqdm import tqdm

# directory = './data/visualization_sample_range/'
# storage_direcroey = './data/visualization_sample_range/'
# # USalign结构比对==========
# exe_path = './data/exe/USalign'
# cluster_center_pdb = './data/elife_cluster_center.pdb'
# # USalign AAA.pdb cluster.pdb -o AAA
# file = '2vfz.pdb'
# temp1 = os.path.join(directory, file)
# temp2 = os.path.join(storage_direcroey, file.rstrip('.pdb')+"_align")
# subprocess.run([exe_path, temp1, cluster_center_pdb, '-o', temp2], stdout=subprocess.DEVNULL)

CompletedProcess(args=['./data/exe/USalign', './data/visualization_sample_range/2vfz.pdb', './data/elife_cluster_center.pdb', '-o', './data/visualization_sample_range/2vfz_align'], returncode=0)

# 提取local structure
- 调用MaSIF修改版的代码（toolManAYY按照需求进行调整）

In [ ]:
# # 摸索以一个结构为模板探索特征提取方法
# from MaSIF.protonate import protonate
# from MaSIF.computeMSMS import computeMSMS
# from MaSIF.computeCharges import computeCharges, assignChargesToNewMesh
# from MaSIF.computeHydrophobicity import computeHydrophobicity
# from MaSIF.fixmesh import fix_mesh
# from MaSIF.masif_opts import masif_opts
# from MaSIF.compute_normal import compute_normal
# from MaSIF.computeAPBS import computeAPBS
# from MaSIF.save_ply import save_ply
# from MaSIF.read_data_from_surface import read_data_from_surface
# import numpy as np
# from Bio.PDB import *
# import pymesh
# from sklearn.neighbors import KDTree

# pdb_filename = '2vfz_align.pdb'
# pdb_path = './data/visualization_sample_range/'
# temp_path = './data/visualization_sample_range/temp/'
# storage_path = './data/visualization_sample_range/'
# sample_redies = 6
# original_file = pdb_path + pdb_filename
# protonate_file = temp_path + pdb_filename
# ply_filename = temp_path + pdb_filename.replace('.pdb', '.ply')
# storage_filename = storage_path + pdb_filename.replace('.pdb', '.npy')

# # Protonated the pdb structure. 
# protonate(original_file, protonate_file)

# # Compute MSMS of surface w/hydrogens.
# vertices1, faces1, normals1, names1, areas1 = computeMSMS(protonate_file, protonate=True)
# # Compute "charged" vertices
# vertex_hbond = computeCharges(protonate_file, vertices1, names1)
# # For each surface residue, assign the hydrophobicity of its amino acid. 
# vertex_hphobicity = computeHydrophobicity(names1)

# vertices2 = vertices1
# faces2 = faces1

# # Fix the mesh.
# mesh = pymesh.form_mesh(vertices2, faces2)
# mesh_original = mesh # ==========测试
# regular_mesh = fix_mesh(mesh, masif_opts['mesh_res'])
# mesh_fixed = regular_mesh # ==========测试
# # Compute the normals
# vertex_normal = compute_normal(regular_mesh.vertices, regular_mesh.faces)

# # Assign charges on new vertices based on charges of old vertices (nearest neighbor)
# vertex_hbond = assignChargesToNewMesh(regular_mesh.vertices, vertices1, vertex_hbond, masif_opts)
# vertex_hphobicity = assignChargesToNewMesh(regular_mesh.vertices, vertices1, vertex_hphobicity, masif_opts)

# vertex_charges = computeAPBS(regular_mesh.vertices, protonate_file, protonate_file.rstrip('.pdb'))

# iface = np.zeros(len(regular_mesh.vertices))
# # Compute the surface of the entire complex and from that compute the interface.
# v3, f3, _, _, _ = computeMSMS(protonate_file, protonate=True)
# # Regularize the mesh
# mesh = pymesh.form_mesh(v3, f3)
# # I believe It is not necessary to regularize the full mesh. This can speed up things by a lot.
# full_regular_mesh = mesh
# # Find the vertices that are in the iface.
# v3 = full_regular_mesh.vertices
# # Find the distance between every vertex in regular_mesh.vertices and those in the full complex.
# kdt = KDTree(v3)
# d, r = kdt.query(regular_mesh.vertices)
# d = np.square(d) # Square d, because this is how it was in the pyflann version.
# assert(len(d) == len(regular_mesh.vertices))
# iface_v = np.where(d >= 2.0)[0]
# iface[iface_v] = 1.0
# # Convert to ply and save.
# save_ply(ply_filename, regular_mesh.vertices, regular_mesh.faces, normals=vertex_normal, \
#          charges=vertex_charges, normalize_charges=True, hbond=vertex_hbond, \
#             hphob=vertex_hphobicity, iface=iface)



In [ ]:
# # ==========Coumpute Data==========

# params = masif_opts['ligand']

# output_dict = {} # vertice_number, xyz, neigh_indecies, si, ddc, hbond, charge, hphob, rho, theta

# # Compute shape complementarity between the two proteins.
# rho = {}
# neigh_indices = {}
# mask = {}
# input_feat = {}
# theta = {}
# iface_labels = {}
# verts = {}

# input_feat, rho, theta, mask, neigh_indices, iface_labels, verts, faces = read_data_from_surface(ply_filename, params)

# UD2_points = np.array([[5.592,16.612,8.001],[6.816,15.928,5.527],[9.885,11.033,5.073],[10.480,12.393,4.845],
#                        [11.608,12.536,4.405],[9.001,12.112,9.408],[9.915,12.481,10.442],[9.347,12.910,8.162],
#                        [8.622,14.137,8.125],[10.824,13.268,8.144],[11.137,13.932,9.366],[11.157,14.239,7.022],
#                        [11.552,15.485,7.610],[9.989,14.500,6.074],[9.684,13.419,5.139],[3.479,12.039,12.290],
#                        [8.745,14.856,6.887],[7.597,14.640,6.076],[4.570,14.080,13.033],[3.973,15.363,13.417],
#                        [4.432,17.575,8.080],[7.685,17.129,5.804],[4.564,16.635,13.189],[3.534,13.457,12.117],
#                        [5.688,16.767,12.658],[6.980,17.066,8.374],[6.329,15.627,4.129],[4.009,13.803,10.725],
#                        [3.919,17.747,13.562],[5.571,15.920,6.544],[3.675,12.764,9.803],[2.715,17.687,14.135],
#                        [5.514,13.946,10.877],[2.133,18.739,14.482],[5.760,14.223,12.256],[2.113,16.459,14.363],
#                        [6.072,15.097,10.058],[5.250,15.344,8.924],[2.764,15.300,13.979]])

# distances = np.full((verts.shape[0],), False, dtype=bool)
# for point in UD2_points:
#    distances_temp = np.sqrt(np.sum((verts - point) ** 2, axis=1))
#    distances_temp = distances_temp < sample_redies
#    distances = distances | distances_temp


# # ==================== 获取边的信息 ====================
# # 创建一个空的边集来存储边，使用集合来避免重复边
# edges = set()
# # 遍历每个面，将其顶点连接成边
# for face in faces:
#     # 获取每个面的三条边，顶点索引两两组合
#     edges.add(tuple(sorted([face[0], face[1]])))
#     edges.add(tuple(sorted([face[1], face[2]])))
#     edges.add(tuple(sorted([face[2], face[0]])))
# # 将边转换为numpy数组
# edges = np.array(list(edges))

# # 使用np.where获取值为True的元素的索引
# true_indices = np.where(distances)[0]
# len(true_indices)
# # 创建一个从0开始的索引列表，这里的长度与true_indices相同
# new_indices = np.arange(len(true_indices))
# # 创建映射关系，将原始索引映射到新的索引
# index_mapping = {original_index: new_index for original_index, new_index in zip(true_indices, new_indices)}

# # 拿到新索引的边
# local_edge = []
# for e in edges:
#     if e[0] in true_indices and e[1] in true_indices:
#         local_edge.append([index_mapping[e[0]], index_mapping[e[1]]])


# output_dict['vertice_number'] = int(verts.shape[0])
# output_dict['index_mapping'] = index_mapping
# output_dict['distances'] = distances
# output_dict['xyz'] = verts[distances, :]
# output_dict['edges'] = np.array(local_edge)
# output_dict['neigh_indecies'] = neigh_indices
# output_dict['si'] = input_feat[:, :, 0][distances, :]
# output_dict['ddc'] = input_feat[:, :, 1][distances, :]
# output_dict['hbond'] = input_feat[:, :, 2][distances, :]
# output_dict['charge'] = input_feat[:, :, 3][distances, :]
# output_dict['hphob'] = input_feat[:, :, 4][distances, :]
# output_dict['rho'] = rho[distances, :]
# output_dict['theta'] = theta[distances, :]

# # Save data only if everything went well. 
# np.save(storage_filename, output_dict)
# print(f"Finished extract features from the strucutr {pdb_filename.rstrip('.pdb')}. Thanks for your using!")

Finished extract features from the strucutr 2vfz_align. Thanks for your using!


# 提取表面网格表示
- 调用之前写的代码

In [ ]:
# # 可视化结构，三个，分别是原始的结构、fixed后的结构、local site后的结构。
# import os

# def write_vertices_to_pdb(filename, vertices):
#     f = open(filename, 'w')
#     step = 0
#     for v in vertices:
#         step += 1
#         line = ""
#         line = line + f"{'ATOM':<6}"
#         line = line + f"{step:>5}"
#         line = line + f"  {'C':<4}"
#         line = line + f"{'ALA':<4}"
#         line = line + f"{'A':<1}"
#         line = line + f"{1:>4}"
#         line = line + f"{round(v[0], 3):>12}"
#         line = line + f"{round(v[1], 3):>8}"
#         line = line + f"{round(v[2], 3):>8}"
#         line = line + f"{1.00:>6}"
#         line = line + f"{95.00:>6}"
#         line = line + f"{'C':>12}"
#         line = line + f"{'  ':>2}\n"
#         f.write(line)
#     f.close()

# storage_path = './data/visualization_sample_range/'
# write_vertices_to_pdb(os.path.join(storage_path, 'grid_original.pdb'), mesh_original.vertices)
# write_vertices_to_pdb(os.path.join(storage_path, 'grid_fixed.pdb'), mesh_fixed.vertices)
# write_vertices_to_pdb(os.path.join(storage_path, 'grid_local.pdb'), verts[distances, :])

# 提取local structure（优化）
- 调用MaSIF修改版的代码（toolManAYY按照需求进行调整）

In [ ]:
# # 摸索以一个结构为模板探索特征提取方法
# from MaSIF.protonate import protonate
# from MaSIF.computeMSMS import computeMSMS
# from MaSIF.computeCharges import computeCharges, assignChargesToNewMesh
# from MaSIF.computeHydrophobicity import computeHydrophobicity
# from MaSIF.fixmesh import fix_mesh
# from MaSIF.masif_opts import masif_opts
# from MaSIF.compute_normal import compute_normal
# from MaSIF.computeAPBS import computeAPBS
# from MaSIF.save_ply import save_ply
# from MaSIF.read_data_from_surface import read_data_from_surface
# import numpy as np
# from Bio.PDB import *
# import pymesh
# from sklearn.neighbors import KDTree

# pdb_filename = '2vfz_align.pdb'
# pdb_path = './data/visualization_sample_range/'
# temp_path = './data/visualization_sample_range/temp/'
# storage_path = './data/visualization_sample_range/'
# sample_redies_1 = 6
# sample_redies_2 = 10
# original_file = pdb_path + pdb_filename
# protonate_file = temp_path + pdb_filename
# ply_filename = temp_path + pdb_filename.replace('.pdb', '.ply')
# storage_filename = storage_path + pdb_filename.replace('.pdb', '_large_sample.npy')

# # # Protonated the pdb structure. 
# # protonate(original_file, protonate_file)

# # # Compute MSMS of surface w/hydrogens.
# # vertices1, faces1, normals1, names1, areas1 = computeMSMS(protonate_file, protonate=True)
# # # Compute "charged" vertices
# # vertex_hbond = computeCharges(protonate_file, vertices1, names1)
# # # For each surface residue, assign the hydrophobicity of its amino acid. 
# # vertex_hphobicity = computeHydrophobicity(names1)

# # vertices2 = vertices1
# # faces2 = faces1

# # # Fix the mesh.
# # mesh = pymesh.form_mesh(vertices2, faces2)
# # mesh_original = mesh # ==========测试
# # regular_mesh = fix_mesh(mesh, masif_opts['mesh_res'])
# # mesh_fixed = regular_mesh # ==========测试
# # # Compute the normals
# # vertex_normal = compute_normal(regular_mesh.vertices, regular_mesh.faces)

# # # Assign charges on new vertices based on charges of old vertices (nearest neighbor)
# # vertex_hbond = assignChargesToNewMesh(regular_mesh.vertices, vertices1, vertex_hbond, masif_opts)
# # vertex_hphobicity = assignChargesToNewMesh(regular_mesh.vertices, vertices1, vertex_hphobicity, masif_opts)

# # vertex_charges = computeAPBS(regular_mesh.vertices, protonate_file, protonate_file.rstrip('.pdb'))

# # iface = np.zeros(len(regular_mesh.vertices))
# # # Compute the surface of the entire complex and from that compute the interface.
# # v3, f3, _, _, _ = computeMSMS(protonate_file, protonate=True)
# # # Regularize the mesh
# # mesh = pymesh.form_mesh(v3, f3)
# # # I believe It is not necessary to regularize the full mesh. This can speed up things by a lot.
# # full_regular_mesh = mesh
# # # Find the vertices that are in the iface.
# # v3 = full_regular_mesh.vertices
# # # Find the distance between every vertex in regular_mesh.vertices and those in the full complex.
# # kdt = KDTree(v3)
# # d, r = kdt.query(regular_mesh.vertices)
# # d = np.square(d) # Square d, because this is how it was in the pyflann version.
# # assert(len(d) == len(regular_mesh.vertices))
# # iface_v = np.where(d >= 2.0)[0]
# # iface[iface_v] = 1.0
# # # Convert to ply and save.
# # save_ply(ply_filename, regular_mesh.vertices, regular_mesh.faces, normals=vertex_normal, \
# #          charges=vertex_charges, normalize_charges=True, hbond=vertex_hbond, \
# #             hphob=vertex_hphobicity, iface=iface)


In [ ]:
# # ==========Coumpute Data==========

# params = masif_opts['ligand']

# output_dict = {} # vertice_number, xyz, neigh_indecies, si, ddc, hbond, charge, hphob, rho, theta

# # Compute shape complementarity between the two proteins.
# rho = {}
# neigh_indices = {}
# mask = {}
# input_feat = {}
# theta = {}
# iface_labels = {}
# verts = {}

# input_feat, rho, theta, mask, neigh_indices, iface_labels, verts, faces = read_data_from_surface(ply_filename, params)

# UDP_points = np.array([[5.592,16.612, 8.001],[6.816,15.928, 5.527],[3.479,12.039,12.290],[4.570,14.080,13.033],
#                        [3.973,15.363,13.417],[4.432,17.575, 8.080],[7.685,17.129, 5.804],[4.564,16.635,13.189],
#                        [3.534,13.457,12.117],[5.688,16.767,12.658],[6.980,17.066, 8.374],[6.329,15.627, 4.129],
#                        [4.009,13.803,10.725],[3.919,17.747,13.562],[5.571,15.920, 6.544],[3.675,12.764, 9.803],
#                        [2.715,17.687,14.135],[5.514,13.946,10.877],[2.133,18.739,14.482],[5.760,14.223,12.256],
#                        [2.113,16.459,14.363],[6.072,15.097,10.058],[5.250,15.344, 8.924],[2.764,15.300,13.979]])
# SUGAR_points = np.array([[ 9.885,11.033, 5.073],[10.480,12.393, 4.845],[11.608,12.536, 4.405],[ 9.001,12.112, 9.408],
#                          [ 9.915,12.481,10.442],[ 9.347,12.910, 8.162],[ 8.622,14.137, 8.125],[10.824,13.268, 8.144],
#                          [11.137,13.932, 9.366],[11.157,14.239, 7.022],[11.552,15.485, 7.610],[ 9.989,14.500, 6.074],
#                          [ 9.684,13.419, 5.139],[ 8.745,14.856, 6.887],[ 7.597,14.640, 6.076]])

# distances = np.full((verts.shape[0],), False, dtype=bool)
# for point in UDP_points:
#    distances_temp = np.sqrt(np.sum((verts - point) ** 2, axis=1))
#    distances_temp = distances_temp < sample_redies_1
#    distances = distances | distances_temp
# for point in SUGAR_points:
#    distances_temp = np.sqrt(np.sum((verts - point) ** 2, axis=1))
#    distances_temp = distances_temp < sample_redies_2
#    distances = distances | distances_temp

# # ==================== 获取边的信息 ====================
# # 创建一个空的边集来存储边，使用集合来避免重复边
# edges = set()
# # 遍历每个面，将其顶点连接成边
# for face in faces:
#     # 获取每个面的三条边，顶点索引两两组合
#     edges.add(tuple(sorted([face[0], face[1]])))
#     edges.add(tuple(sorted([face[1], face[2]])))
#     edges.add(tuple(sorted([face[2], face[0]])))
# # 将边转换为numpy数组
# edges = np.array(list(edges))

# # 使用np.where获取值为True的元素的索引
# true_indices = np.where(distances)[0]
# # 创建一个从0开始的索引列表，这里的长度与true_indices相同
# new_indices = np.arange(len(true_indices))
# # 创建映射关系，将原始索引映射到新的索引
# index_mapping = {original_index: new_index for original_index, new_index in zip(true_indices, new_indices)}

# # 拿到新索引的边
# local_edge = []
# for e in edges:
#     if e[0] in true_indices and e[1] in true_indices:
#         local_edge.append([index_mapping[e[0]], index_mapping[e[1]]])


# output_dict['vertice_number'] = int(verts.shape[0])
# output_dict['index_mapping'] = index_mapping
# output_dict['distances'] = distances
# output_dict['xyz'] = verts[distances, :]
# output_dict['edges'] = np.array(local_edge)
# output_dict['neigh_indecies'] = neigh_indices
# output_dict['si'] = input_feat[:, :, 0][distances, :]
# output_dict['ddc'] = input_feat[:, :, 1][distances, :]
# output_dict['hbond'] = input_feat[:, :, 2][distances, :]
# output_dict['charge'] = input_feat[:, :, 3][distances, :]
# output_dict['hphob'] = input_feat[:, :, 4][distances, :]
# output_dict['rho'] = rho[distances, :]
# output_dict['theta'] = theta[distances, :]

# # Save data only if everything went well. 
# # np.save(storage_filename, output_dict)
# print(f"Finished extract features from the strucutr {pdb_filename.rstrip('.pdb')}. Thanks for your using!")

Finished extract features from the strucutr 2vfz_align. Thanks for your using!


# 提取表面网格表示（优化）
- 调用之前写的代码

In [ ]:
# # 可视化结构，三个，分别是原始的结构、fixed后的结构、local site后的结构。
# import os

# def write_vertices_to_pdb(filename, vertices):
#     f = open(filename, 'w')
#     step = 0
#     for v in vertices:
#         step += 1
#         line = ""
#         line = line + f"{'ATOM':<6}"
#         line = line + f"{step:>5}"
#         line = line + f"  {'C':<4}"
#         line = line + f"{'ALA':<4}"
#         line = line + f"{'A':<1}"
#         line = line + f"{1:>4}"
#         line = line + f"{round(v[0], 3):>12}"
#         line = line + f"{round(v[1], 3):>8}"
#         line = line + f"{round(v[2], 3):>8}"
#         line = line + f"{1.00:>6}"
#         line = line + f"{95.00:>6}"
#         line = line + f"{'C':>12}"
#         line = line + f"{'  ':>2}\n"
#         f.write(line)
#     f.close()

# storage_path = './data/visualization_sample_range/'
# # write_vertices_to_pdb(os.path.join(storage_path, 'grid_large_original.pdb'), mesh_original.vertices)
# # write_vertices_to_pdb(os.path.join(storage_path, 'grid_large_fixed.pdb'), mesh_fixed.vertices)
# write_vertices_to_pdb(os.path.join(storage_path, 'grid_large_local_6_10.pdb'), verts[distances, :])